In [ ]:
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from surya.foundation import FoundationPredictor
from surya.layout import LayoutPredictor
from surya.settings import settings


def apply_otsu_threshold(image: Image.Image) -> Image.Image:
    """
    Apply Otsu's automatic thresholding to a grayscale PIL Image.

    Args:
        image: Grayscale PIL Image

    Returns:
        Thresholded PIL Image
    """
    img_array = np.array(image)
    _, thresholded = cv2.threshold(img_array, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return Image.fromarray(thresholded)

IMAGE_PATH = "/home/bas/Documents/Visual Code Data/BelHisHAAI/1909 - jpeg/EHC_B665_O_2025_1909_III_0105.jpg"


def display_predictions(image: Image.Image, predictions, output_path: str = None) -> Image.Image:
    """
    Display layout predictions on the image by drawing bounding boxes and labels.

    Args:
        image: PIL Image to annotate
        predictions: Layout predictions from Surya's LayoutPredictor
        output_path: Optional path to save the annotated image

    Returns:
        Annotated PIL Image
    """
    # Create a copy to avoid modifying the original, convert to RGB for drawing
    annotated = image.copy().convert("RGB")
    draw = ImageDraw.Draw(annotated)

    # Color map for different layout types
    colors = {
        "Text": "blue",
        "Title": "red",
        "List": "green",
        "Table": "purple",
        "Figure": "orange",
        "Caption": "cyan",
        "Header": "magenta",
        "Footer": "brown",
        "PageHeader": "magenta",
        "PageFooter": "brown",
        "SectionHeader": "darkred",
        "Footnote": "gray",
        "Formula": "pink",
    }
    default_color = "yellow"

    # Try to load a font, fall back to default if not available
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 16)
    except (IOError, OSError):
        font = ImageFont.load_default()

    # Draw each prediction
    for pred in predictions.bboxes:
        bbox = pred.bbox  # [x1, y1, x2, y2]
        label = pred.label
        confidence = pred.confidence if hasattr(pred, 'confidence') else None

        color = colors.get(label, default_color)

        # Draw the bounding box
        draw.rectangle(bbox, outline=color, width=3)

        # Create label text
        label_text = label
        if confidence is not None:
            label_text = f"{label} ({confidence:.2f})"

        # Draw label background and text
        # Position label above the box, or inside if too close to top edge
        label_y = bbox[1] - 20
        if label_y < 0:
            label_y = bbox[1] + 5  # Draw inside the box instead

        text_bbox = draw.textbbox((bbox[0], label_y), label_text, font=font)
        draw.rectangle(text_bbox, fill=color)
        draw.text((bbox[0], label_y), label_text, fill="white", font=font)

    # Save if output path provided
    if output_path:
        annotated.save(output_path)
        print(f"Saved annotated image to {output_path}")

    return annotated


image = Image.open(IMAGE_PATH).convert("L")
image = apply_otsu_threshold(image)
layout_predictor = LayoutPredictor(FoundationPredictor(checkpoint=settings.LAYOUT_MODEL_CHECKPOINT))

# layout_predictions is a list of dicts, one per image
layout_predictions = layout_predictor([image])

# DEBUG: Check predictions
print(f"Found {len(layout_predictions[0].bboxes)} bboxes")
for pred in layout_predictions[0].bboxes:
    print(f"  {pred.label}: {pred.bbox}")

# Display predictions on the image
annotated_image = display_predictions(image, layout_predictions[0], "result_surya_ocr.jpg")
annotated_image.show()


ModuleNotFoundError: No module named 'surya'